# PRism gate classifier (Kaggle)

Fine-tunes CodeBERT on the CodeReviewer quality-estimation data, picks the threshold for about
90% recall and exports ONNX. All logic lives in `training/gate/` in the repo; this notebook only
runs it. Settings: Accelerator GPU T4, Internet on. Details in `training/README.md`.

## 1. Settings

Change the branch or commit here. `MAX_TRAIN_ROWS` is a random sample of the train split; `None` trains on all of it.

In [ ]:
REPO = "ManvGit17/prism"
REF = "phase-3"  # branch, tag or commit
MAX_TRAIN_ROWS = 100000  # None = all 265,836 rows; time per step on a T4 not measured
WORK = "/kaggle/temp"  # big scratch space; not saved with the notebook
OUT = "/kaggle/working/gate_out"

## 2. Secrets

Reads the secrets added under Add-ons, Secrets. `GITHUB_TOKEN` is needed to clone the private
repo; `WANDB_API_KEY` and `HF_TOKEN` are optional. Only yes or no is printed, never a value.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

_client = UserSecretsClient()


def secret(name):
    try:
        return _client.get_secret(name) or None
    except Exception:
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
for name in ("WANDB_API_KEY", "HF_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value
print("GITHUB_TOKEN set:", bool(GITHUB_TOKEN))
print("WANDB_API_KEY set:", "WANDB_API_KEY" in os.environ)
print("HF_TOKEN set:", "HF_TOKEN" in os.environ)
assert GITHUB_TOKEN, (
    "Add a GITHUB_TOKEN secret (read access to the repo) and attach it to this notebook."
)

## 3. Clone the repo

The token goes to git as a request header through environment variables. It is not in the URL, not
in `.git/config`, and any git output is scrubbed before it is printed.

In [ ]:
import base64
import subprocess

_basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
_env = dict(
    os.environ,
    GIT_CONFIG_COUNT="1",
    GIT_CONFIG_KEY_0="http.extraheader",
    GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}",
    GIT_TERMINAL_PROMPT="0",
)


def scrub(text):
    return text.replace(GITHUB_TOKEN, "***").replace(_basic, "***")


def git(*args, cwd=None):
    proc = subprocess.run(["git", *args], cwd=cwd, env=_env, capture_output=True, text=True)
    print(scrub(proc.stdout + proc.stderr).strip())
    if proc.returncode:
        raise SystemExit(f"git {args[0]} failed")


os.chdir("/kaggle/working")
if not os.path.isdir("prism"):
    git("clone", "--quiet", f"https://github.com/{REPO}.git", "prism")
os.chdir("/kaggle/working/prism")
git("checkout", "--quiet", REF)
git("log", "-1", "--oneline")

## 4. Install requirements

Installs the small list in `training/requirements-kaggle.txt`. torch is left as Kaggle provides it.

In [ ]:
!pip install -q -r training/requirements-kaggle.txt
import torch

print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 5. Download the dataset

Gets `Diff_Quality_Estimation.zip` (2.8 GB) from Zenodo. It is read in place, never unpacked.

In [ ]:
import zipfile

DATA = f"{WORK}/Diff_Quality_Estimation.zip"
URL = "https://zenodo.org/records/6900648/files/Diff_Quality_Estimation.zip?download=1"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(DATA):
    !curl -L --fail --retry 5 -o {DATA} "{URL}"
with zipfile.ZipFile(DATA) as zf:
    for info in zf.infolist():
        print(info.filename, info.file_size)

## 6. Dry run

A few hundred rows through the whole pipeline (train, threshold, export, parity) with the real
model. It should take a few minutes and prove the setup works. The metrics here mean nothing.

In [ ]:
!python -m training.gate.run --data {DATA} --out {WORK}/dry \
    --head --max-train-rows 400 --max-valid-rows 200 --max-test-rows 200 \
    --epochs 1 --eval-steps 10 --logging-steps 5

## 7. Real run

The full pipeline. Run it only after the dry run passed. Progress is logged in the cell output.

In [ ]:
rows = f"--max-train-rows {MAX_TRAIN_ROWS}" if MAX_TRAIN_ROWS else ""
!python -m training.gate.run --data {DATA} --out {WORK}/run {rows}

## 8. Collect the outputs

Copies the ONNX model, `gate_config.json`, the tokenizer files and `metrics.json` into
`/kaggle/working/gate_out`, which Kaggle saves as notebook output.

In [ ]:
import shutil

shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(f"{WORK}/run/onnx", OUT)
shutil.copy(f"{WORK}/run/metrics.json", OUT)
for name in sorted(os.listdir(OUT)):
    print(name, os.path.getsize(os.path.join(OUT, name)))

## 9. Results

Prints the threshold section of `metrics.json` to copy into `training/model_cards/gate.md`.

In [ ]:
import json

with open(f"{OUT}/metrics.json") as fh:
    m = json.load(fh)
print(
    json.dumps({k: m[k] for k in ("threshold", "class_balance", "onnx_parity_max_diff")}, indent=2)
)